# DQN：深度 Q 网络——从 Q 表到神经网络的跨越
> 难度标签：高级 | 预计时长：20-30分钟 | 前置知识：无学习经验


> 所属模块：05_强化学习 | 源文件：DQN.py | 核心功能：经验回放、目标网络、epsilon 衰减、PyTorch 实现

## 概述

DQN（Deep Q-Network）是 DeepMind 2015 年在 Nature 上发表的里程碑工作。它用神经网络替代 Q 表，配合经验回放和目标网络两大技巧，在 Atari 游戏上达到了超越人类的水平。

Q-Learning 的 Q 表在连续状态空间中不可行（状态数无穷多），DQN 用一个神经网络 Q(s,a;theta) 来近似 Q 函数。

脚本在自实现的 CartPole 环境中训练 DQN，展示了经验回放、目标网络和训练过程。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
# --- 导入库 ---
import random

## 数学原理

### 1. DQN 的核心思想

**代码对应**：DQN 用神经网络 $Q(s, a; \theta)$ 近似 Q 函数。

Q-Learning 的表格法在状态空间大或连续时不可行。DQN 用深度网络参数化 Q 函数：

$$Q(s, a; \theta) \approx Q^*(s, a)$$

### 2. 损失函数

$$L(\theta) = \mathbb{E}_{(s, a, r, s') \sim \mathcal{D}}\left[\left(r + \gamma\max_{a'}Q(s', a'; \theta^{-}) - Q(s, a; \theta)\right)^2\right]$$

其中 $\theta^{-}$ 为目标网络参数（定期从 $\theta$ 复制），$\mathcal{D}$ 为经验回放缓冲区。

### 3. 经验回放（Experience Replay）

**代码对应**：经验回放打破数据间的时间相关性。

存储转移 $(s, a, r, s')$ 到缓冲区 $\mathcal{D}$，训练时随机采样 mini-batch：

$$\theta \leftarrow \theta - \alpha\nabla_\theta L(\theta)$$

梯度：

$$\nabla_\theta L = \mathbb{E}\left[2(r + \gamma\max_{a'}Q(s', a'; \theta^-) - Q(s, a; \theta))\nabla_\theta Q(s, a; \theta)\right]$$

### 4. 目标网络（Target Network）

直接用 $Q(s', a'; \theta)$ 计算 TD 目标会导致"追逐移动目标"的不稳定。目标网络 $\theta^-$ 定期从 $\theta$ 复制：

$$y = r + \gamma\max_{a'}Q(s', a'; \theta^-)$$

这使训练目标在一定时期内保持稳定。

### 5. $\epsilon$-贪婪探索

$$a = \begin{cases}\arg\max_a Q(s, a; \theta) & \text{with prob } 1-\epsilon \\ \text{random action} & \text{with prob } \epsilon\end{cases}$$

$\epsilon$ 通常从 1.0 线性衰减到 0.01。

### 1. 简单 CartPole 环境（自实现）

运行 1. 简单 CartPole 环境（自实现） 的代码，观察算法在该环节的行为。

In [ ]:
class SimpleCartPole:
    """简化版 CartPole 环境，状态: [位置, 速度, 角度, 角速度]"""
    def __init__(self):
        self.max_steps = 200
        self.reset()

    def reset(self):
        self.state = np.random.uniform(-0.05, 0.05, 4)
        self.steps = 0
        return self.state.copy()

    def step(self, action):
        x, x_dot, theta, theta_dot = self.state
        force = 1.0 if action == 1 else -1.0
        # 物理模拟（简化）
        x_ddot = (force - 0.0025 * x_dot + 0.001 * np.sin(theta)) / 1.0
        theta_ddot = (0.01 * np.sin(theta) - 0.001 * theta_dot + 0.0001 * force) / 0.1

        dt = 0.02
        x_dot += x_ddot * dt
        theta_dot += theta_ddot * dt
        x += x_dot * dt
        theta += theta_dot * dt

        self.state = np.array([x, x_dot, theta, theta_dot])
        self.steps += 1

        done = abs(x) > 2.4 or abs(theta) > 0.21 or self.steps >= self.max_steps
        reward = 1.0 if not done else 0.0
        return self.state.copy(), reward, done

### 2. DQN 网络

运行 2. DQN 网络 的代码，观察算法在该环节的行为。

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
# --- 继续 ---
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return self.net(x)

### 3. 经验回放

运行 3. 经验回放 的代码，观察算法在该环节的行为。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))

    def __len__(self):
        return len(self.buffer)

### 4. DQN Agent

运行 4. DQN Agent 的代码，观察算法在该环节的行为。

In [ ]:
class DQNAgent:
    def __init__(self, state_dim=4, action_dim=2, lr=1e-3, gamma=0.99,
                 epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01,
                 batch_size=32, target_update=10):
        self.action_dim = action_dim
# --- 赋值/配置 ---
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.batch_size = batch_size
# --- 赋值/配置 ---
        self.target_update = target_update

        self.q_net = QNetwork(state_dim, action_dim)
        self.target_net = QNetwork(state_dim, action_dim)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
# --- 赋值/配置 ---
        self.losses = []

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
# --- 继续 ---
            q_vals = self.q_net(state_t)
            return q_vals.argmax(dim=1).item()

    def update(self):
        if len(self.buffer) < self.batch_size:
            return

        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        states_t = torch.FloatTensor(states)
        actions_t = torch.LongTensor(actions).unsqueeze(1)
        rewards_t = torch.FloatTensor(rewards)
        next_states_t = torch.FloatTensor(next_states)
        dones_t = torch.FloatTensor(dones)

        # 当前 Q 值
        q_values = self.q_net(states_t).gather(1, actions_t).squeeze()

        # 目标 Q 值（使用目标网络）
        with torch.no_grad():
            next_q = self.target_net(next_states_t).max(dim=1)[0]
            target = rewards_t + self.gamma * next_q * (1 - dones_t)

        loss = nn.MSELoss()(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        self.losses.append(loss.item())

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

### 5. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
env = SimpleCartPole()
agent = DQNAgent()

print("=== DQN 训练 ===")
n_episodes = 300
reward_history = []

for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.buffer.push(state, action, reward, next_state, done)
        agent.update()
# --- 赋值/配置 ---
        state = next_state
        total_reward += reward

    agent.decay_epsilon()
    if (episode + 1) % agent.target_update == 0:
        agent.update_target()

    reward_history.append(total_reward)

    if (episode + 1) % 50 == 0:
        avg = np.mean(reward_history[-50:])
        avg_loss = np.mean(agent.losses[-100:]) if agent.losses else 0
        print(f"  Episode {episode+1:>3}: 平均奖励={avg:>6.1f}, "
              f"ε={agent.epsilon:.3f}, 平均Loss={avg_loss:.4f}")

### 6. 测试

运行 6. 测试 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 测试（关闭探索）===")
agent.epsilon = 0
test_rewards = []
for _ in range(10):
    state = env.reset()
# --- 赋值/配置 ---
    total_reward = 0
    done = False
    while not done:
        action = agent.select_action(state)
        state, reward, done = env.step(action)
# --- 赋值/配置 ---
        total_reward += reward
    test_rewards.append(total_reward)
print(f"10 次测试平均奖励: {np.mean(test_rewards):.1f}")

### 7. DQN 关键组件

运行 7. DQN 关键组件 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== DQN 关键组件 ===")
print("1. 经验回放 (Experience Replay): 打破数据相关性，提高样本效率")
print("2. 目标网络 (Target Network): 冻结目标 Q 值，防止训练振荡")
print("3. ε-greedy 探索: 初期大量探索，逐步趋向利用")
print("4. 神经网络近似: 用函数 Q(s,a;θ) 近似 Q 表格，处理连续状态空间")

print("\n=== DQN 要点 ===")
print("- 解决了 Q-Learning 在连续状态空间的局限")
print("- 训练不稳定：需要目标网络、经验回放、梯度裁剪等技巧")
print("- 只适合离散动作空间（连续动作用 DDPG/PPO）")
print("- Atari 游戏中表现惊人（仅用像素输入）")

## 关键代码解释

### 两大创新

**经验回放**：打破时间相关性，让数据可以复用。

```python
buffer.push(state, action, reward, next_state, done)
batch = buffer.sample(batch_size)  # 随机采样一批经验
```

**目标网络**：冻结目标 Q 值，防止训练振荡。

```python
target = rewards + gamma * target_net(next_states).max(dim=1)[0] * (1 - dones)
# 每隔 target_update 步同步一次
target_net.load_state_dict(q_net.state_dict())
```

### 损失函数

```python
q_values = q_net(states).gather(1, actions)
loss = MSELoss(q_values, target)
```

## 使用示例

```python
agent = DQNAgent(state_dim=4, action_dim=2)
for episode in range(n_episodes):
    state = env.reset()
    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.buffer.push(state, action, reward, next_state, done)
        agent.update()
        state = next_state
```

## 注意事项

1. **只适合离散动作**：连续动作空间用 DDPG/SAC
2. **训练不稳定**：需要目标网络、经验回放、梯度裁剪
3. **样本效率低**：需要大量交互数据
4. **过估计偏差**：max 操作会高估 Q 值，Double DQN 可缓解

## 总结与延伸

以上代码展示了 **DQN** 的完整流程。

**解读要点**：
- 关注 **累计奖励** 随训练轮次的增长趋势
- 观察 **探索率 epsilon** 的衰减过程
- 对比不同策略（greedy vs epsilon-greedy）的表现

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Double DQN**：用在线网络选动作，目标网络评估，减少过估计
- **Dueling DQN**：将 Q 值分解为 V(s) + A(s,a)
- **Prioritized Experience Replay**：优先采样 TD 误差大的经验
- **Rainbow DQN**：融合多种 DQN 变体的集大成者
